### ЗАДАЧА: Панель онбординга поставщиков по паттерну `MVC`

Команда vendor onboarding в маркетплейсе проверяет новые компании перед тем,
как открыть им доступ к публикации товаров и первым выплатам.
Для каждого кейса нужно хранить:
- `application_id` — уникальный идентификатор заявки;
- `company` — название компании;
- `country` — страна регистрации;
- `category` — основная товарная категория;
- `status` — текущий статус заявки;
- `manager` — менеджер, который ведет кейс;
- `documents_ok` — получены ли документы;
- `risk_flag` — есть ли риск-флаг на дополнительную проверку;
- `decision` — финальное решение по заявке.

Нужно реализовать консольную внутреннюю панель по паттерну `MVC`, где:
- `Model` хранит заявки и бизнес-правила;
- `View` отвечает только за отображение данных и сообщений;
- `Controller` принимает действия оператора и связывает `Model` и `View`.

### Жизненный цикл заявки

Заявка проходит через такие состояния:
- `new` — новая заявка, только создана;
- `docs_pending` — у компании запросили документы;
- `docs_received` — документы получены и приняты системой;
- `under_review` — заявка ушла в ручную проверку менеджером;
- `risk_review` — по заявке поднят риск-флаг и нужна дополнительная проверка;
- `approved` — финально одобрена;
- `rejected` — финально отклонена.

### Бизнес-правила

Система должна работать по следующим правилам:
- нельзя создать заявку с уже существующим `application_id`;
- нельзя назначить менеджера несуществующей заявке;
- финальную заявку (`approved`, `rejected`) нельзя менять дальше;
- документы можно запросить только у заявки в статусе `new`;
- отметить документы как полученные можно только из статуса `docs_pending`;
- переход в `under_review` возможен только если:
  - у заявки назначен `manager`;
  - `documents_ok == True`;
  - текущий статус равен `docs_received`;
- поставить риск-флаг можно только из статуса `under_review`;
- снять риск-флаг можно только из статуса `risk_review`, при этом:
  - `risk_flag` должен стать `False`;
  - статус должен вернуться в `under_review`;
- одобрить заявку можно только из статуса `under_review`;
- заявку нельзя одобрять, пока `risk_flag == True`;
- отклонить заявку можно только из статуса `under_review` или `risk_review`.

### Нужно привести реализацию к корректному поведению:
- добавить недостающие проверки и ограничения;
- исправить переходы между статусами;
- не допускать изменений финальных заявок;
- сохранить разделение на `Model`, `View`, `Controller`;
- оставить тот же формат данных и сценарий действий из `actions`;
- обеспечить вывод списка заявок и сводки по статусам.

In [7]:
from dataclasses import dataclass


initial_applications = [
    ("VO-100", "best-gadgets", "DE", "electronics"),
    ("VO-101", "home-decor-plus", "PL", "home"),
]

actions = [
    ("show",),
    ("assign", "VO-100", "Olga"),
    ("request_docs", "VO-100"),
    ("review", "VO-100"),
    ("docs_received", "VO-100"),
    ("review", "VO-100"),
    ("flag_risk", "VO-100"),
    ("approve", "VO-100"),
    ("clear_risk", "VO-100"),
    ("approve", "VO-100"),
    ("create", "VO-102", "fresh-market", "IT", "grocery"),
    ("assign", "VO-102", "Max"),
    ("docs_received", "VO-102"),
    ("request_docs", "VO-102"),
    ("docs_received", "VO-102"),
    ("review", "VO-102"),
    ("reject", "VO-102"),
    ("show",),
]


@dataclass
class VendorApplication:
    application_id: str
    company: str
    country: str
    category: str
    status: str = "new"
    manager: str | None = None
    documents_ok: bool = False
    risk_flag: bool = False
    decision: str | None = None


class VendorOnboardingModel:
    def __init__(self):
        self.applications = {}
        self.done_stats = {'approved','rejected'}

    def add_application(self, application_id: str, company: str, country: str, category: str) -> None:
        if application_id in self.applications:
            raise ValueError('aplication already exists')
        self.applications[application_id] = VendorApplication(application_id, company, country, category)
        
        

    def assign_manager(self, application_id: str, manager: str) -> None:
        if application_id not in self.applications:
            raise ValueError("Application not found")
        self.applications[application_id].manager = manager

    def request_documents(self, application_id: str) -> None:
        if application_id not in self.applications:
            raise ValueError("Application not found")
        if self.applications[application_id].status in self.done_stats:
            raise ValueError('already done')
        if self.applications[application_id].status != 'new':
            raise ValueError('must be new')

        self.applications[application_id].status = "docs_pending"

    def mark_documents_received(self, application_id: str) -> None:
        if self.applications[application_id].status in self.done_stats:
            raise ValueError('already done')
        if self.applications[application_id].status != 'docs_pending':
            raise ValueError('status must be pending')

        if application_id not in self.applications:
            raise ValueError("Application not found")
        application = self.applications[application_id]
        application.documents_ok = True
        application.status = "docs_received"

    def start_review(self, application_id: str) -> None:
        if self.applications[application_id].status in self.done_stats:
            raise ValueError('already done')

        if application_id not in self.applications:
            raise ValueError("Application not found")
        application = self.applications[application_id]
        if application.manager is None:
            raise ValueError("Manager is required")
        if application.documents_ok != True:
            raise ValueError('documents are not ok')
        if application.status != 'docs_received':
            raise ValueError('not docs_received')

        application.status = "under_review"

    def flag_risk(self, application_id: str) -> None:
        if self.applications[application_id].status in self.done_stats:
            raise ValueError('already done')
            

        if application_id not in self.applications:
            raise ValueError("Application not found")
        
        if self.applications[application_id].status != 'under_review':
            raise ValueError('not under_review' )

        application = self.applications[application_id]
        application.risk_flag = True
        application.status = "risk_review"

    def clear_risk(self, application_id: str) -> None:
        if self.applications[application_id].status in self.done_stats:
            raise ValueError('already done')
        if self.applications[application_id].status != 'risk_review':
            raise ValueError('not risk_review' )

        if application_id not in self.applications:
            raise ValueError("Application not found")
        application = self.applications[application_id]
        application.risk_flag = False
        application.status = "under_review"

    def approve_application(self, application_id: str) -> None:
        if self.applications[application_id].status in self.done_stats:
            raise ValueError('already done')
        if self.applications[application_id].status != 'under_review':
            raise ValueError('not under_review')
        if self.applications[application_id].risk_flag == True:
            raise ValueError('cant be approved, risk flag True')

        if application_id not in self.applications:
            raise ValueError("Application not found")
        application = self.applications[application_id]
        application.status = "approved"
        application.decision = "approved"

    def reject_application(self, application_id: str) -> None:
        if self.applications[application_id].status in self.done_stats:
            raise ValueError('already done')
        
        if self.applications[application_id].status not in {'under_review', 'risk_review'}:
            raise ValueError('cant reject')
        
        if application_id not in self.applications:
            raise ValueError("Application not found")
        application = self.applications[application_id]
        application.status = "rejected"
        application.decision = "rejected"

    def list_applications(self) -> list[str]:
        rows = []
        for application in self.applications.values():
            rows.append(
                f"{application.application_id} | {application.company} | {application.country} | "
                f"{application.category} | {application.status} | {application.manager} | "
                f"{application.documents_ok} | {application.risk_flag} | {application.decision}"
            )
        return rows

    def status_summary(self) -> dict[str, int]:
        summary = {}
        for application in self.applications.values():
            summary[application.status] = summary.get(application.status, 0) + 1
        return summary


class VendorOnboardingView:
    @staticmethod
    def render_applications(rows: list[str]) -> None:
        print("Vendor applications:")
        for row in rows:
            print(row)

    @staticmethod
    def render_summary(summary: dict[str, int]) -> None:
        print("Status summary:", summary)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class VendorOnboardingController:
    def __init__(self, model: VendorOnboardingModel, view: VendorOnboardingView):
        self.model = model
        self.view = view

    def create_application(self, application_id: str, company: str, country: str, category: str) -> None:
        try:
            self.model.add_application(application_id, company, country, category)
            self.view.render_success(f"Application {application_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_manager(self, application_id: str, manager: str) -> None:
        try:
            self.model.assign_manager(application_id, manager)
            self.view.render_success(f"Manager assigned to {application_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def request_documents(self, application_id: str) -> None:
        try:
            self.model.request_documents(application_id)
            self.view.render_success(f"Documents requested for {application_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def mark_documents_received(self, application_id: str) -> None:
        try:
            self.model.mark_documents_received(application_id)
            self.view.render_success(f"Documents received for {application_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_review(self, application_id: str) -> None:
        try:
            self.model.start_review(application_id)
            self.view.render_success(f"Application {application_id} moved to review")
        except ValueError as error:
            self.view.render_error(str(error))

    def flag_risk(self, application_id: str) -> None:
        try:
            self.model.flag_risk(application_id)
            self.view.render_success(f"Risk flag enabled for {application_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def clear_risk(self, application_id: str) -> None:
        try:
            self.model.clear_risk(application_id)
            self.view.render_success(f"Risk flag cleared for {application_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def approve_application(self, application_id: str) -> None:
        try:
            self.model.approve_application(application_id)
            self.view.render_success(f"Application {application_id} approved")
        except ValueError as error:
            self.view.render_error(str(error))

    def reject_application(self, application_id: str) -> None:
        try:
            self.model.reject_application(application_id)
            self.view.render_success(f"Application {application_id} rejected")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_applications(self) -> None:
        self.view.render_applications(self.model.list_applications())
        self.view.render_summary(self.model.status_summary())


model = VendorOnboardingModel()
view = VendorOnboardingView()
controller = VendorOnboardingController(model, view)

for application_id, company, country, category in initial_applications:
    model.add_application(application_id, company, country, category)

for action in actions:
    if action[0] == "show":
        controller.show_applications()
    elif action[0] == "create":
        _, application_id, company, country, category = action
        controller.create_application(application_id, company, country, category)
    elif action[0] == "assign":
        _, application_id, manager = action
        controller.assign_manager(application_id, manager)
    elif action[0] == "request_docs":
        _, application_id = action
        controller.request_documents(application_id)
    elif action[0] == "docs_received":
        _, application_id = action
        controller.mark_documents_received(application_id)
    elif action[0] == "review":
        _, application_id = action
        controller.start_review(application_id)
    elif action[0] == "flag_risk":
        _, application_id = action
        controller.flag_risk(application_id)
    elif action[0] == "clear_risk":
        _, application_id = action
        controller.clear_risk(application_id)
    elif action[0] == "approve":
        _, application_id = action
        controller.approve_application(application_id)
    elif action[0] == "reject":
        _, application_id = action
        controller.reject_application(application_id)

print()
print("Финальное состояние:")
controller.show_applications()


Vendor applications:
VO-100 | best-gadgets | DE | electronics | new | None | False | False | None
VO-101 | home-decor-plus | PL | home | new | None | False | False | None
Status summary: {'new': 2}
SUCCESS: Manager assigned to VO-100
SUCCESS: Documents requested for VO-100
ERROR: documents are not ok
SUCCESS: Documents received for VO-100
SUCCESS: Application VO-100 moved to review
SUCCESS: Risk flag enabled for VO-100
ERROR: not under_review
SUCCESS: Risk flag cleared for VO-100
SUCCESS: Application VO-100 approved
SUCCESS: Application VO-102 created
SUCCESS: Manager assigned to VO-102
ERROR: status must be pending
SUCCESS: Documents requested for VO-102
SUCCESS: Documents received for VO-102
SUCCESS: Application VO-102 moved to review
SUCCESS: Application VO-102 rejected
Vendor applications:
VO-100 | best-gadgets | DE | electronics | approved | Olga | True | False | approved
VO-101 | home-decor-plus | PL | home | new | None | False | False | None
VO-102 | fresh-market | IT | grocery 

In [ ]:
# Vendor applications:
# VO-100 | best-gadgets | DE | electronics | new | None | False | False | None
# VO-101 | home-decor-plus | PL | home | new | None | False | False | None
# Status summary: {'new': 2}
# SUCCESS: Manager assigned to VO-100
# SUCCESS: Documents requested for VO-100
# ERROR: Documents are required
# SUCCESS: Documents received for VO-100
# SUCCESS: Application VO-100 moved to review
# SUCCESS: Risk flag enabled for VO-100
# ERROR: Application is not under review
# SUCCESS: Risk flag cleared for VO-100
# SUCCESS: Application VO-100 approved
# SUCCESS: Application VO-102 created
# SUCCESS: Manager assigned to VO-102
# ERROR: Application is not waiting for documents
# SUCCESS: Documents requested for VO-102
# SUCCESS: Documents received for VO-102
# SUCCESS: Application VO-102 moved to review
# SUCCESS: Application VO-102 rejected
# Vendor applications:
# VO-100 | best-gadgets | DE | electronics | approved | Olga | True | False | approved
# VO-101 | home-decor-plus | PL | home | new | None | False | False | None
# VO-102 | fresh-market | IT | grocery | rejected | Max | True | False | rejected
# Status summary: {'approved': 1, 'new': 1, 'rejected': 1}
#
# Финальное состояние:
# Vendor applications:
# VO-100 | best-gadgets | DE | electronics | approved | Olga | True | False | approved
# VO-101 | home-decor-plus | PL | home | new | None | False | False | None
# VO-102 | fresh-market | IT | grocery | rejected | Max | True | False | rejected
# Status summary: {'approved': 1, 'new': 1, 'rejected': 1}